# PINN — Boussinesq VBM: all five benchmark cases (Kaggle)

Runs `pinn_boussinesq_benchmarks.ipynb` once per case in a **fresh namespace**, collects the
results and writes a summary.

**Why a driver rather than a loop inside the benchmarks notebook.** `compute_residuals` and
`compute_loss` are `@tf.function`s that bake case constants — bathymetry, VBM coefficients,
the sponge and breaking switches, the boundary-condition branch — into their traced graph at
first call. Changing a Python global afterwards does **not** invalidate that cache, so a naive
in-notebook loop would silently run case 2 on case 1's compiled graph. Re-executing the whole
notebook in a fresh namespace per case sidesteps that completely, and keeps the benchmarks
notebook as the single verified source of truth rather than duplicating physics here.

Each case is independent: if one raises, the traceback is recorded and the batch continues.

## 0. Knobs

In [ ]:
import os, sys, json, time, traceback, subprocess
from pathlib import Path

# Printed first and stored in results.json, so a log always says which version ran.
# Kaggle re-runs the notebook saved in its editor, not the file on disk.
NOTEBOOK_VERSION = "2026-08-15  five-case VBM batch: CG exact, sponge, breaking closure"

QUICK = False          # True -> ~3 min smoke test over all five cases (trains nothing
                       # useful, but exercises every physics assert and every plot path)

REQUIRE_GPU = True     # stop before training if no GPU is visible.  QUICK is exempt.
                       # On CPU the full batch is roughly 8-15x slower.

SHOW_FIGURES = True    # False -> suppress inline figures; the PNGs are still written
                       # to each case's output directory.

# Cheapest and most diagnostic first, so a systematic problem surfaces in minutes
# rather than after the first long train.
CASE_QUEUE = [
    "flat_cosine",           # ~2 min   sanity check, flat bottom, periodic
    "carrier_greenspan",     # ~25 min  the only case with an exact solution
    "solitary_nonbreaking",  # ~12 min  Synolakis H/d = 0.0185
    "solitary_breaking",     # ~18 min  Synolakis H/d = 0.3, eddy-viscosity breaking
    "beji_battjes",          # ~22 min  bar + sponge layer + wave-maker
]

# Repeat ONE case across several seeds instead of running the queue, to measure
# how much of a run-to-run difference is real.  Comparisons here have been
# single realisations, and an accidental control (flat_cosine, whose sampling
# was unchanged between two runs) still moved its final loss by 48% purely from
# a shifted RNG stream.  Set to None for the normal queue.
SEED_SWEEP = ("carrier_greenspan", [0, 1, 2])

# Applied on top of each case's own config.  QUICK trims the training only -- every
# verification assert in the notebook still runs at full strength.
QUICK_OVERRIDES = dict(epochs_adam=200, epochs_lbfgs=30,
                       N_F=1500, N_IC=150, N_BC=150, plot_freq=100)

REPO_URL = "https://github.com/phoenixfin/sciml-framework.git"
                       # cloned only if the benchmarks notebook is not already
                       # visible; needs Kaggle "Internet" on.  Set "" if you
                       # attach the repo as a Dataset instead.

print(NOTEBOOK_VERSION)
print(f"QUICK={QUICK}  REQUIRE_GPU={REQUIRE_GPU}  cases={len(CASE_QUEUE)}")

## 0.1 Locate the benchmarks notebook

Same discovery strategy as `kaggle_swe_revision_all`: cheap exact checks against the working
directory and its parents, then a bounded walk of every plausible Kaggle root, then a shallow
clone as a last resort.

In [ ]:
TARGET = "pinn_boussinesq_benchmarks.ipynb"
SUBDIRS = ("", "notebooks/pinn_boussinesq")


def _walk_dirs(root, max_depth=5):
    root = Path(root)
    if not root.is_dir():
        return
    skip = {".git", "__pycache__", ".ipynb_checkpoints", "node_modules",
            ".pytest_cache", ".ruff_cache", "site-packages"}
    frontier = [(root, 0)]
    while frontier:
        d, depth = frontier.pop(0)
        yield d
        if depth >= max_depth:
            continue
        try:
            frontier.extend((c, depth + 1) for c in d.iterdir()
                            if c.is_dir() and c.name not in skip)
        except OSError:
            continue


def find_bench(extra_roots=()):
    here = Path.cwd().resolve()
    for s in [Path(os.environ["SCIML_REPO"])] if os.environ.get("SCIML_REPO") else []:
        for sub in SUBDIRS:
            cand = (s / sub if sub else s) / TARGET
            if cand.is_file():
                return cand.resolve()
    for s in [here, *here.parents]:
        for sub in SUBDIRS:
            cand = (s / sub if sub else s) / TARGET
            if cand.is_file():
                return cand.resolve()
    roots = [Path(r) for r in extra_roots] + [
        Path("/kaggle/input"), Path("/kaggle/working"), Path("/kaggle/usr/lib"), here]
    for root in roots:
        for d in _walk_dirs(root):
            if (d / TARGET).is_file():
                return (d / TARGET).resolve()
    return None


BENCH_NB = find_bench()
clone_note = "not attempted (already present)" if BENCH_NB else "not attempted"

if BENCH_NB is None and REPO_URL:
    # clone OUTSIDE the output directory, or the checkout is served up as run output
    base = next((Path(p) for p in ("/kaggle/temp", "/tmp") if Path(p).is_dir()), Path.cwd())
    dest = base / "sciml_repo"
    if dest.exists():
        clone_note = f"{dest} already exists, reused"
    else:
        print(f"cloning {REPO_URL} -> {dest} ...")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)],
                           capture_output=True, text=True)
        clone_note = ("ok" if r.returncode == 0
                      else f"FAILED (exit {r.returncode}): {(r.stderr or '').strip()[-400:]}")
        print("git clone:", clone_note)
    BENCH_NB = find_bench([dest])

print("clone:", clone_note)
if BENCH_NB is None:
    raise SystemExit(
        f"could not find {TARGET}. Attach the repo as a Kaggle Dataset, enable "
        f"Internet so the clone can run, or set SCIML_REPO to the checkout.")
print("benchmarks notebook:", BENCH_NB)

# Which commit are we actually testing?  The driver clones from GitHub, so an
# uncommitted local fix is invisible here -- print the SHA rather than guess.
_repo = BENCH_NB.parent
while _repo != _repo.parent and not (_repo / ".git").exists():
    _repo = _repo.parent
if (_repo / ".git").exists():
    _sha = subprocess.run(["git", "-C", str(_repo), "log", "-1", "--format=%h %s"],
                          capture_output=True, text=True)
    print("  commit:", (_sha.stdout or _sha.stderr).strip()[:100])
else:
    print("  commit: (not a git checkout)")

_nb = json.loads(BENCH_NB.read_text(encoding="utf-8"))
_code = [c for c in _nb["cells"] if c["cell_type"] == "code"]
print(f"  {len(_nb['cells'])} cells, {len(_code)} of them code")
_cases = [k for k in ("flat_cosine", "carrier_greenspan", "solitary_nonbreaking",
                      "solitary_breaking", "beji_battjes")
          if f'"{k}": {{' in "".join("".join(c["source"]) for c in _code)]
print("  cases defined:", _cases)
missing = [c for c in CASE_QUEUE if c not in _cases]
assert not missing, f"CASE_QUEUE names not present in the notebook: {missing}"

## 0.2 Environment and output directories

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf

GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:                       # avoid grabbing all VRAM up front
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("memory growth:", e)

print(f"TensorFlow {tf.__version__}   NumPy {np.__version__}")
print(f"GPU: {[g.name for g in GPUS] if GPUS else 'NONE (CPU only)'}")
if REQUIRE_GPU and not GPUS and not QUICK:
    raise SystemExit(
        "No GPU visible and REQUIRE_GPU is set. Enable the GPU accelerator, "
        "or set REQUIRE_GPU=False to accept a much slower CPU run.")

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
os.chdir(OUT_DIR)                    # the benchmarks notebook writes to ./pinn_results/<case>
RESULTS_DIR = OUT_DIR / "pinn_results"
RESULTS_DIR.mkdir(exist_ok=True)
print("output root:", OUT_DIR)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 9})
_T0 = time.time()


def banner(title):
    print("\n" + "=" * 78)
    print(f"  {title}    [t+{time.time() - _T0:7.1f}s]")
    print("=" * 78)


_real_show = plt.show
if not SHOW_FIGURES:
    plt.show = lambda *a, **k: plt.close("all")

est = {"flat_cosine": 2, "carrier_greenspan": 25, "solitary_nonbreaking": 12,
       "solitary_breaking": 18, "beji_battjes": 22}
total = sum(est.get(c, 15) for c in CASE_QUEUE)
print(f"rough GPU estimate for this queue: {total} min"
      + ("  (QUICK: ~3 min)" if QUICK else "")
      + ("" if GPUS else "   -- on CPU expect 8-15x that"))

## 1. Run the queue

Each case gets a namespace containing only `ACTIVE_CASE` and `CFG_OVERRIDES`; the notebook
supplies everything else. `clear_session()` between cases drops the previous graph so five
runs in one process do not accumulate VRAM.

In [ ]:
def run_case(case, overrides, seed=None):
    """Execute the benchmarks notebook end to end for one case."""
    ns = {"__name__": "__main__",
          "ACTIVE_CASE": case,
          "CFG_OVERRIDES": dict(overrides)}
    if seed is not None:
        ns["SEED"] = int(seed)
        ns["SAVE_SUFFIX"] = f"_seed{int(seed)}"   # keep the runs from overwriting
    cells = json.loads(BENCH_NB.read_text(encoding="utf-8"))["cells"]
    n_code = 0
    for k, cell in enumerate(cells):
        if cell["cell_type"] != "code":
            continue
        n_code += 1
        code = "".join(cell["source"])
        exec(compile(code, f"<{case}:cell{k}>", "exec"), ns)
    assert "CASE_RESULT" in ns, "notebook finished without publishing CASE_RESULT"
    return ns["CASE_RESULT"], n_code


results, failures = {}, {}
overrides = QUICK_OVERRIDES if QUICK else {}

if SEED_SWEEP:
    _c, _seeds = SEED_SWEEP
    JOBS = [(f"{_c}#seed{s}", _c, s) for s in _seeds]
    print(f"seed sweep: {_c} x {len(_seeds)} seeds {list(_seeds)}")
else:
    JOBS = [(c, c, None) for c in CASE_QUEUE]

for label, case, seed in JOBS:
    banner(f"CASE {label}")
    tf.keras.backend.clear_session()
    t0 = time.time()
    try:
        res, n_code = run_case(case, overrides, seed)
        res["wall_seconds"] = round(time.time() - t0, 1)
        res["quick"] = bool(QUICK)
        results[label] = res
        print(f"\n[{label}] OK in {res['wall_seconds']:.1f}s "
              f"({n_code} code cells), final loss {res['loss_final']:.3e}")
    except Exception:
        tb = traceback.format_exc()
        failures[label] = tb
        print(f"\n[{label}] FAILED after {time.time() - t0:.1f}s\n{tb}")
        print("continuing with the remaining cases")

plt.show = _real_show
banner(f"queue finished: {len(results)} ok, {len(failures)} failed")

## 2. Summary

In [ ]:
hdr = (f"{'case':<22} {'status':>7} {'wall [s]':>9} {'final loss':>12} "
       f"{'PDE':>10} {'IC':>10} {'BC':>10}")
print(hdr); print("-" * len(hdr))
for case in [j[0] for j in JOBS]:
    if case in results:
        r = results[case]
        print(f"{case:<22} {'ok':>7} {r['wall_seconds']:>9.1f} {r['loss_final']:>12.3e} "
              f"{r['loss_pde']:>10.2e} {r['loss_ic']:>10.2e} {r['loss_bc']:>10.2e}")
    else:
        print(f"{case:<22} {'FAILED':>7} {'-':>9} {'-':>12} {'-':>10} {'-':>10} {'-':>10}")

# ---- spread across seeds -------------------------------------------------
if SEED_SWEEP and len(results) > 1:
    keys = [("loss_final", "final loss", "{:.3e}"),
            ("corr_eta",   "r(eta)",     "{:.4f}"),
            ("corr_u",     "r(u)",       "{:.4f}"),
            ("rmse_eta_rel", "RMSE/amp", "{:.4f}"),
            ("runup_pinn", "run-up [m]", "{:.5f}")]
    print("\nSpread across seeds — this is the noise floor for every comparison:")
    print(f"  {'metric':<12}{'mean':>12}{'std':>12}{'min':>12}{'max':>12}{'CV':>9}")
    print("  " + "-" * 69)
    for k, lbl, fmt in keys:
        vals = [r[k] for r in results.values() if k in r]
        if len(vals) < 2:
            continue
        m = float(np.mean(vals)); s = float(np.std(vals, ddof=1))
        cv = abs(s / m) if m else float("nan")
        print(f"  {lbl:<12}{fmt.format(m):>12}{fmt.format(s):>12}"
              f"{fmt.format(min(vals)):>12}{fmt.format(max(vals)):>12}{cv:>8.1%}")
    print("  A change smaller than about 2 std is not evidence of anything.")

_cg = next((k for k in results if k.startswith("carrier_greenspan")), None)
if _cg:
    r = results[_cg]
    print("\nCarrier-Greenspan vs the exact solution "
          "(model gap floor is ~0.7%, CG is non-dispersive):")
    print(f"  eta  RMSE = {r['rmse_eta']:.4e} m  ({100*r['rmse_eta_rel']:.2f}% of amplitude)"
          f"   Pearson r = {r['corr_eta']:.4f}")
    print(f"  u    RMSE = {r['rmse_u']:.4e} m/s                        "
          f"   Pearson r = {r['corr_u']:.4f}")
    print(f"  run-up: exact {r['runup_exact']:.4f} m   PINN {r['runup_pinn']:.4f} m   "
          f"({100*abs(r['runup_pinn']-r['runup_exact'])/r['runup_exact']:.1f}% error)")

summary = {
    "notebook_version": NOTEBOOK_VERSION,
    "quick": bool(QUICK),
    "gpu": [g.name for g in GPUS],
    "tensorflow": tf.__version__,
    "queue": CASE_QUEUE,
    "wall_seconds_total": round(time.time() - _T0, 1),
    "results": results,
    "failures": {k: v.splitlines()[-1] for k, v in failures.items()},
}
with open(RESULTS_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"\nsummary -> {RESULTS_DIR / 'results.json'}")

for case in CASE_QUEUE:
    d = RESULTS_DIR / case
    if d.is_dir():
        files = sorted(p.name for p in d.iterdir())
        print(f"  {case:<22} {len(files):>2} files: {', '.join(files[:6])}"
              + (" ..." if len(files) > 6 else ""))

if failures:
    print("\n" + "!" * 78)
    print(f"{len(failures)} case(s) failed: {', '.join(failures)}")
    print("Full tracebacks are above; the last line of each is in results.json.")
    print("!" * 78)